# Baseline: ARMA for Long-Term Time Series Forecasting

## 0. Get Setup

In [2]:
# Clone the github repo and download dependencies
!git clone https://github.com/chunyagi/PatchTST.git

Cloning into 'PatchTST'...
remote: Enumerating objects: 431, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 431 (delta 16), reused 14 (delta 14), pack-reused 406 (from 2)
Receiving objects: 100% (431/431), 12.97 MiB | 30.73 MiB/s, done.
Resolving deltas: 100% (199/199), done.


## 1. Download Data

In [3]:
# Download data from google drive with gdown
try:
    import gdown
    print(f"[INFO] gdown version: {gdown.__version__}")
except:
    print("[INFO] Couldn't find gdown, installing it...")
    !pip install -q gdown
    import gdown
    print(f"[INFO] gdown version: {gdown.__version__}")

# ETTh1
!mkdir -p /data/datasets/public/ETDataset/ETT-small
!gdown 1vOClm_t4RgUf8nqherpTfNnB8rrYq14Q -O /data/datasets/public/ETDataset/ETT-small/

# Weather
!mkdir -p /data/datasets/public/weather/
!gdown 1Tc7GeVN7DLEl-RAs-JVwG9yFMf--S8dy -O /data/datasets/public/weather/

[INFO] gdown version: 5.2.1
Downloading...
From: https://drive.google.com/uc?id=1vOClm_t4RgUf8nqherpTfNnB8rrYq14Q
To: /data/datasets/public/ETDataset/ETT-small/ETTh1.csv
100% 2.59M/2.59M [00:00<00:00, 205MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Tc7GeVN7DLEl-RAs-JVwG9yFMf--S8dy
To: /data/datasets/public/weather/weather.csv
100% 7.24M/7.24M [00:00<00:00, 54.9MB/s]


## 2. Data Preprocessing

In [4]:
# Create directory
!mkdir -p /content/arma_baseline

In [5]:
%%writefile /content/arma_baseline/data_utils.py

import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

def _get_data(args):
    """Load and preprocess data for ARMA baseline."""
    # Read CSV and set date as index
    df = pd.read_csv(os.path.join(args.root_path, args.data_path))
    df['date'] = pd.to_datetime(df.iloc[:, 0])
    df.set_index('date', inplace=True)

    # Dataset-specific boundaries
    if 'ETTh' in args.data:
        # ETTh1 / ETTh2 (hourly)
        border1s = [0, 12*30*24 - args.seq_len, 12*30*24 + 4*30*24 - args.seq_len]
        border2s = [12*30*24, 12*30*24 + 4*30*24, 12*30*24 + 8*30*24]
    elif 'ETTm' in args.data:
        # ETTm1 / ETTm2 (15-min)
        border1s = [0, 12*30*24*4 - args.seq_len, 12*30*24*4 + 4*30*24*4 - args.seq_len]
        border2s = [12*30*24*4, 12*30*24*4 + 4*30*24*4, 12*30*24*4 + 8*30*24*4]
    else:
        num_train = int(len(df) * args.train_split)
        num_test = int(len(df) * args.test_split)
        num_vali = len(df) - num_train - num_test
        border1s = [0, num_train - args.seq_len, len(df) - num_test - args.seq_len]
        border2s = [num_train, num_train + num_vali, len(df)]

    # Get train and test dataframes
    train_df = df.iloc[border1s[0]:border2s[0]]
    test_df = df.iloc[border1s[2]:border2s[2]]

    scaler = StandardScaler()
    scaler.fit(train_df.values)

    # Standardize as DataFrames (keeps index/structure)
    train_df = pd.DataFrame(
        scaler.transform(train_df.values),
        index=train_df.index,
        columns=train_df.columns
    )
    test_df = pd.DataFrame(
        scaler.transform(test_df.values),
        index=test_df.index,
        columns=test_df.columns
    )

    # Calculate seasonal means on standardized data
    hourly_means = train_df.groupby(train_df.index.hour).mean()

    if args.use_monthly:
        monthly_means = train_df.groupby(train_df.index.month).mean()
        monthly_means = monthly_means.reindex(range(1, 13), method='ffill')
    else:
        monthly_means = None

    # Deseasonalize train set
    hourly_seasonal_train = pd.DataFrame([hourly_means.loc[d.hour] for d in train_df.index],
                                        index=train_df.index)
    train_deseasonalized = train_df - hourly_seasonal_train

    if args.use_monthly:
        monthly_seasonal_train = pd.DataFrame([monthly_means.loc[d.month] for d in train_df.index],
                                              index=train_df.index)
        train_deseasonalized = train_deseasonalized - monthly_seasonal_train

    # Deseasonalize test set
    hourly_seasonal_test = pd.DataFrame([hourly_means.loc[d.hour] for d in test_df.index],
                                        index=test_df.index)
    test_deseasonalized = test_df - hourly_seasonal_test

    if args.use_monthly:
        monthly_seasonal_test = pd.DataFrame([monthly_means.loc[d.month] for d in test_df.index],
                                            index=test_df.index)
        test_deseasonalized = test_deseasonalized - monthly_seasonal_test

    # Convert to numpy arrays
    train_data = train_deseasonalized.values
    test_data = test_deseasonalized.values

    # Create test windows
    test_windows = []
    for i in range(len(test_data) - args.seq_len - args.pred_len + 1):
        context = test_data[i:i + args.seq_len]
        target = test_data[i + args.seq_len:i + args.seq_len + args.pred_len]
        test_windows.append((context, target))

    # Subsample if requested
    if args.subsample_test < 1.0:
        step = int(1 / args.subsample_test)
        test_windows = test_windows[::step]
        print(f"Subsampling to {args.subsample_test*100}% of test windows: {len(test_windows)} windows")

    return train_data, test_windows, hourly_means, monthly_means

Writing /content/arma_baseline/data_utils.py


In [6]:
# from types import SimpleNamespace

# args = SimpleNamespace(
#     root_path='/data/datasets/public/ETDataset/ETT-small',
#     data_path='ETTh1.csv',
#     data='ETTh1',
#     seq_len=512,
#     pred_len=96,
#     deseasonalize=True,
#     train_split=0.7,
#     test_split=0.2
# )

# train_data, test_windows, hourly_means, monthly_means = _get_data(args)
# print(f"Train shape: {train_data.shape}")
# print(f"Num test windows: {len(test_windows)}")
# print(f"Hourly means shape: {hourly_means.shape}")
# print(f"Monthly means shape: {monthly_means.shape}")

## 3. Helper Functions

In [7]:
%%writefile /content/arma_baseline/helpers.py

import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
import os

def grid_search_arma(train_data, max_p=5, max_q=5, plot_fits=False):
    """
    Grid search over ARMA(p,q) orders using AIC.

    Args:
        train_data: [train_len, n_vars] - fit separate ARMA per variable
        max_p: maximum AR order
        max_q: maximum MA order

    Returns:
        best_orders: list of (p,q) tuples, one per variable
    """
    n_vars = train_data.shape[1]
    best_orders = []
    fitted_values_all = []

    for var_idx in range(n_vars):
        series = train_data[:, var_idx]
        best_aic = np.inf
        best_bic = np.inf
        best_order_aic = (0, 0)
        best_order_bic = (0, 0)
        best_fitted_values = None

        print(f"Grid searching for variable {var_idx + 1}/{n_vars}...")

        for p in range(max_p + 1):
            for q in range(max_q + 1):
                if p == 0 and q == 0:
                    continue  # Skip (0,0) - no model

                try:
                    model = ARIMA(series, order=(p, 0, q))  # d=0 for ARMA
                    fitted = model.fit()

                    if fitted.aic < best_aic:
                        best_aic = fitted.aic
                        best_order_aic = (p, q)

                    if fitted.bic < best_bic:
                        best_bic = fitted.bic
                        best_order_bic = (p, q)
                        best_fitted_values = fitted.fittedvalues

                except:
                    # Skip if model fails to converge
                    continue

        # Use BIC
        best_orders.append(best_order_bic)
        fitted_values_all.append(best_fitted_values)

        print(f"  Variable {var_idx + 1}: Best order = ARMA{best_order_bic} (BIC={best_bic:.2f}), AIC suggested {best_order_aic}")

    # Visualize fits
    if plot_fits:
        print("\nSaving individual training fit plots...")
        os.makedirs('arma_fits', exist_ok=True)

        for var_idx in range(n_vars):
            plt.figure(figsize=(10, 4))
            plt.plot(train_data[:, var_idx], label='Actual', alpha=0.7, linewidth=1)
            plt.plot(fitted_values_all[var_idx], label='Fitted', alpha=0.7, linewidth=1)
            plt.title(f'Variable {var_idx + 1}: ARMA{best_orders[var_idx]}')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(f'arma_fits/var_{var_idx+1}.png', dpi=150)
            plt.close()

    print(f"Saved {n_vars} plots to arma_fits/ folder")

    return best_orders


def forecast_arma(context, best_orders, pred_len):
    """
    Forecast using ARMA by refitting on each test window.

    Args:
        context: [seq_len, n_vars] - context window
        best_orders: list of (p,q) tuples per variable
        pred_len: number of steps to forecast

    Returns:
        predictions: [pred_len, n_vars]
    """
    n_vars = context.shape[1]
    predictions = np.zeros((pred_len, n_vars))

    for var_idx in range(n_vars):
        series = context[:, var_idx]
        p, q = best_orders[var_idx]

        try:
            # Fit ARMA on context window
            model = ARIMA(series, order=(p, 0, q))
            fitted = model.fit()

            # Forecast pred_len steps
            forecast = fitted.forecast(steps=pred_len)
            predictions[:, var_idx] = forecast

        except:
            # If fitting fails, use naive forecast (last value)
            print(f"Warning: ARMA{best_orders[var_idx]} failed to converge for variable {var_idx + 1}. Using last value as forecast.")
            predictions[:, var_idx] = series[-1]

    return predictions

Writing /content/arma_baseline/helpers.py


## 4. Training/Evaluation Script

In [8]:
%%writefile /content/arma_baseline/main.py

import argparse
import numpy as np
import pandas as pd
import os
import sys

sys.path.append('/content/arma_baseline/')

from data_utils import _get_data
from helpers import grid_search_arma, forecast_arma


# Main function
def main(args):

    # Create save path
    setting = f'{args.model_id}_{args.data}_sl{args.seq_len}_pl{args.pred_len}'
    save_path = os.path.join(args.checkpoints, setting)
    os.makedirs(save_path, exist_ok=True)

    # Load data
    print('Args in experiment:')
    print(args)

    train_data, test_windows, hourly_means, monthly_means = _get_data(args)
    print(f"Train shape: {train_data.shape}, Test windows: {len(test_windows)}")

    # Grid search or load saved orders
    orders_file = os.path.join(save_path, 'best_orders.npy')
    if os.path.exists(orders_file):
        print(f">>>>>>>Loading saved ARMA orders from {orders_file}>>>>>>>>>>>>>>>>>>>>>>>>>>")
        best_orders = np.load(orders_file, allow_pickle=True).tolist()
    else:
        print(f"\n>>>>>>>Grid searching ARMA orders (max_p={args.max_p}, max_q={args.max_q})>>>>>>>>>>>>>>>>>>>>>>>>>>")
        best_orders = grid_search_arma(train_data, max_p=args.max_p, max_q=args.max_q, plot_fits=args.plot_fits)
        np.save(orders_file, best_orders)
        print(f"Best orders: {best_orders}")

    # Forecast on test windows
    print(f"\n>>>>>>>Forecasting on {len(test_windows)} test windows>>>>>>>>>>>>>>>>>>>>>>>>>>")
    all_predictions = []
    all_targets = []

    for i, (context, target) in enumerate(test_windows):
        if (i + 1) % 100 == 0:
            print(f"  Processing window {i + 1}/{len(test_windows)}")

        pred = forecast_arma(context, best_orders, args.pred_len)
        all_predictions.append(pred)
        all_targets.append(target)

    # Calculate metrics
    all_predictions = np.array(all_predictions)  # [n_windows, pred_len, n_vars]
    all_targets = np.array(all_targets)

    mse = np.mean((all_predictions - all_targets) ** 2)
    mae = np.mean(np.abs(all_predictions - all_targets))

    print(f"\nResults:")
    print(f"MSE: {mse:.6f}")
    print(f"MAE: {mae:.6f}")

    # Save results
    results_df = pd.DataFrame({
        'mse': [mse],
        'mae': [mae],
        'best_orders': [str(best_orders)]
    })
    results_df.to_csv(os.path.join(save_path, 'result.csv'), index=False)
    print(f"\nResults saved to {save_path}/result.csv")


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='ARMA Baseline for Long-term Time Series Forecasting')

    # Data loader
    parser.add_argument('--data', type=str, default='ETTh1', help='dataset type')
    parser.add_argument('--root_path', type=str, default='/data/datasets/public/ETDataset/ETT-small/', help='root path of data file')
    parser.add_argument('--data_path', type=str, default='ETTh1.csv', help='data file')
    parser.add_argument('--subsample_test', type=float, default=1.0, help='fraction of test windows to use (e.g., 0.5 for every 2nd window)')
    parser.add_argument('--checkpoints', type=str, default='./checkpoints/', help='location to save results')

    # Forecasting task
    parser.add_argument('--seq_len', type=int, default=512, help='input sequence length')
    parser.add_argument('--pred_len', type=int, default=96, help='prediction sequence length')

    # ARMA parameters
    parser.add_argument('--max_p', type=int, default=5, help='maximum AR order')
    parser.add_argument('--max_q', type=int, default=5, help='maximum MA order')
    parser.add_argument('--plot_fits', action='store_true', help='save ARMA training fit plots')
    parser.add_argument('--deseasonalize', action='store_true', help='remove seasonality')
    parser.add_argument('--use_monthly', action='store_true', help='include monthly deseasonalization')

    # For non-ETT datasets
    parser.add_argument('--train_split', type=float, default=0.7, help='train split ratio')
    parser.add_argument('--test_split', type=float, default=0.2, help='test split ratio')
    parser.add_argument('--val_split', type=float, default=0.1, help='validation split ratio')

    # Experiment
    parser.add_argument('--model_id', type=str, default='arma_test', help='model id')

    args = parser.parse_args()

    # Evaluation
    main(args)

Writing /content/arma_baseline/main.py


In [9]:
%cd /content/arma_baseline

/content/arma_baseline


## 5. Train Model on ETTh1

### 5.1 Horizon 96

In [ ]:
!python main.py \
    --data ETTh1 \
    --root_path /data/datasets/public/ETDataset/ETT-small/ \
    --data_path ETTh1.csv \
    --seq_len 512 \
    --pred_len 96 \
    --max_p 1 \
    --max_q 1 \
    --plot_fits \
    --deseasonalize \
    --use_monthly \
    --model_id arma_etth1_test

Args in experiment:
Namespace(data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', checkpoints='./checkpoints/', seq_len=512, pred_len=96, max_p=1, max_q=1, plot_fits=True, deseasonalize=True, train_split=0.7, test_split=0.2, val_split=0.1, model_id='arma_etth1_test')
Train shape: (8640, 7), Test windows: 2785

>>>>>>>Grid searching ARMA orders (max_p=1, max_q=1)>>>>>>>>>>>>>>>>>>>>>>>>>>
Grid searching for variable 1/7...
  Variable 1: Best order = ARMA(1, 1) (BIC=4858.79), AIC suggested (1, 1)
Grid searching for variable 2/7...
  Variable 2: Best order = ARMA(1, 1) (BIC=4401.61), AIC suggested (1, 1)
Grid searching for variable 3/7...
  Variable 3: Best order = ARMA(1, 1) (BIC=4732.32), AIC suggested (1, 1)
Grid searching for variable 4/7...
  Variable 4: Best order = ARMA(1, 1) (BIC=4138.71), AIC suggested (1, 1)
Grid searching for variable 5/7...
  Variable 5: Best order = ARMA(1, 1) (BIC=10885.75), AIC suggested (1, 1)
Grid searching for var

### 5.2 Horizon 192

In [ ]:
!python main.py \
    --data ETTh1 \
    --root_path /data/datasets/public/ETDataset/ETT-small/ \
    --data_path ETTh1.csv \
    --seq_len 512 \
    --pred_len 192 \
    --max_p 1 \
    --max_q 1 \
    --plot_fits \
    --deseasonalize \
    --use_monthly \
    --model_id arma_etth1_test

Args in experiment:
Namespace(data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', checkpoints='./checkpoints/', seq_len=512, pred_len=192, max_p=1, max_q=1, plot_fits=True, deseasonalize=True, train_split=0.7, test_split=0.2, val_split=0.1, model_id='arma_etth1_test')
Train shape: (8640, 7), Test windows: 2689

>>>>>>>Grid searching ARMA orders (max_p=1, max_q=1)>>>>>>>>>>>>>>>>>>>>>>>>>>
Grid searching for variable 1/7...
  Variable 1: Best order = ARMA(1, 1) (BIC=4858.79), AIC suggested (1, 1)
Grid searching for variable 2/7...
  Variable 2: Best order = ARMA(1, 1) (BIC=4401.61), AIC suggested (1, 1)
Grid searching for variable 3/7...
  Variable 3: Best order = ARMA(1, 1) (BIC=4732.32), AIC suggested (1, 1)
Grid searching for variable 4/7...
  Variable 4: Best order = ARMA(1, 1) (BIC=4138.71), AIC suggested (1, 1)
Grid searching for variable 5/7...
  Variable 5: Best order = ARMA(1, 1) (BIC=10885.75), AIC suggested (1, 1)
Grid searching for va

### 5.3 Horizon 336

In [ ]:
!python main.py \
    --data ETTh1 \
    --root_path /data/datasets/public/ETDataset/ETT-small/ \
    --data_path ETTh1.csv \
    --seq_len 512 \
    --pred_len 336 \
    --max_p 1 \
    --max_q 1 \
    --plot_fits \
    --deseasonalize \
    --use_monthly \
    --model_id arma_etth1_test

Args in experiment:
Namespace(data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', checkpoints='./checkpoints/', seq_len=512, pred_len=336, max_p=1, max_q=1, plot_fits=True, deseasonalize=True, train_split=0.7, test_split=0.2, val_split=0.1, model_id='arma_etth1_test')
Train shape: (8640, 7), Test windows: 2545

>>>>>>>Grid searching ARMA orders (max_p=1, max_q=1)>>>>>>>>>>>>>>>>>>>>>>>>>>
Grid searching for variable 1/7...
  Variable 1: Best order = ARMA(1, 1) (BIC=4858.79), AIC suggested (1, 1)
Grid searching for variable 2/7...
  Variable 2: Best order = ARMA(1, 1) (BIC=4401.61), AIC suggested (1, 1)
Grid searching for variable 3/7...
  Variable 3: Best order = ARMA(1, 1) (BIC=4732.32), AIC suggested (1, 1)
Grid searching for variable 4/7...
  Variable 4: Best order = ARMA(1, 1) (BIC=4138.71), AIC suggested (1, 1)
Grid searching for variable 5/7...
  Variable 5: Best order = ARMA(1, 1) (BIC=10885.75), AIC suggested (1, 1)
Grid searching for va

### 5.4 Horizon 720

In [ ]:
!python main.py \
    --data ETTh1 \
    --root_path /data/datasets/public/ETDataset/ETT-small/ \
    --data_path ETTh1.csv \
    --seq_len 512 \
    --pred_len 720 \
    --max_p 1 \
    --max_q 1 \
    --plot_fits \
    --deseasonalize \
    --use_monthly \
    --model_id arma_etth1_test

Args in experiment:
Namespace(data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', checkpoints='./checkpoints/', seq_len=512, pred_len=720, max_p=1, max_q=1, plot_fits=True, deseasonalize=True, train_split=0.7, test_split=0.2, val_split=0.1, model_id='arma_etth1_test')
Train shape: (8640, 7), Test windows: 2161

>>>>>>>Grid searching ARMA orders (max_p=1, max_q=1)>>>>>>>>>>>>>>>>>>>>>>>>>>
Grid searching for variable 1/7...
  Variable 1: Best order = ARMA(1, 1) (BIC=4858.79), AIC suggested (1, 1)
Grid searching for variable 2/7...
  Variable 2: Best order = ARMA(1, 1) (BIC=4401.61), AIC suggested (1, 1)
Grid searching for variable 3/7...
  Variable 3: Best order = ARMA(1, 1) (BIC=4732.32), AIC suggested (1, 1)
Grid searching for variable 4/7...
  Variable 4: Best order = ARMA(1, 1) (BIC=4138.71), AIC suggested (1, 1)
Grid searching for variable 5/7...
  Variable 5: Best order = ARMA(1, 1) (BIC=10885.75), AIC suggested (1, 1)
Grid searching for va

## 6. Train Model on Weather

### 6.1 Horizon 96

In [ ]:
!python main.py \
    --data weather \
    --root_path /data/datasets/public/weather/ \
    --data_path weather.csv \
    --subsample_test 0.5 \
    --seq_len 512 \
    --pred_len 96 \
    --max_p 1 \
    --max_q 1 \
    --plot_fits \
    --deseasonalize \
    --model_id arma_weather_test

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/mode

### 6.2 Horizon 192

In [9]:
!python main.py \
    --data weather \
    --root_path /data/datasets/public/weather/ \
    --data_path weather.csv \
    --subsample_test 0.5 \
    --seq_len 512 \
    --pred_len 192 \
    --max_p 1 \
    --max_q 1 \
    --plot_fits \
    --deseasonalize \
    --model_id arma_weather_test

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/mode

### 6.3 Horizon 336

In [9]:
!python main.py \
    --data weather \
    --root_path /data/datasets/public/weather/ \
    --data_path weather.csv \
    --subsample_test 0.5 \
    --seq_len 512 \
    --pred_len 336 \
    --max_p 1 \
    --max_q 1 \
    --plot_fits \
    --deseasonalize \
    --model_id arma_weather_test

Streaming output truncated to the last 5000 lines.
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


### 6.4 Horizon 720

In [10]:
!python main.py \
    --data weather \
    --root_path /data/datasets/public/weather/ \
    --data_path weather.csv \
    --subsample_test 0.5 \
    --seq_len 512 \
    --pred_len 720 \
    --max_p 1 \
    --max_q 1 \
    --plot_fits \
    --deseasonalize \
    --model_id arma_weather_test

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/mode